# SVD Gastronomía — Sistema de Recomendación
Sistema de recomendación personalizado de servicios gastronómicos del País Vasco.

**Misma arquitectura que `svd.ipynb` (patrimonio)**, adaptada al dataset de gastronomía:
- `gastro_id` en lugar de `culture_id`
- Intereses gastronómicos (Hostelería, Gourmet, Bodegas, Sidrería...)
- Features específicas: Michelin, Repsol, Denominación de Origen, Entorno (multi-label)
- Afinidad extendida: interés del usuario × tipo de lugar gastronómico


## 1. Imports

In [35]:
import pandas as pd
import numpy as np
import pickle, os, random, warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

from surprise import SVD, Dataset, Reader, accuracy, KNNBasic, KNNWithMeans, NMF, BaselineOnly
from surprise.model_selection import cross_validate, train_test_split as surp_split

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split as sk_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

random.seed(42)
np.random.seed(42)

## 2. Carga de datos

In [36]:
gastro    = pd.read_csv("../data/gastronomia.csv")
qualif    = pd.read_csv("../data/gastronomy_qualifications.csv")
usuarios  = pd.read_csv("../data/usuarios.csv")
intereses = pd.read_csv("../data/intereses.csv")
int_usu   = pd.read_csv("../data/intereses_usuarios.csv")

gastro.rename(columns={"Unnamed: 0": "gastro_id"}, inplace=True)

print(f"Restaurantes/lugares gastro: {len(gastro)}")
print(f"\nTipo de lugar:")
print(gastro["Tipo de lugar"].value_counts())
print(f"\nvaloracion: min={gastro['valoracion'].min()}  max={gastro['valoracion'].max()}  "
      f"std={gastro['valoracion'].std():.3f}")

Restaurantes/lugares gastro: 490

Tipo de lugar:
Tipo de lugar
Restaurante       279
Bodega             65
Sidreria           44
tienda             34
Bodega Txakoli     33
Asador             23
queseria           12
Name: count, dtype: int64

valoracion: min=4.1  max=5.0  std=0.237


## 3. Feature engineering — Gastronomía
Mismo proceso que en `feature_engineering.ipynb`, sección GASTRONOMÍA.

In [ ]:
# ── Limpieza y merge con qualifications ──────────────────────────────────────
COLS_ELIMINAR = ["Nombre","Descripcion","Dirección","Municipio","Email","Teléfono",
                  "WEB","URL amigable","url_imagen","lat","lon","Patrocinado","Active"]

qualif["gastro_id"] = gastro["gastro_id"].values
gastro = gastro.merge(qualif, on="gastro_id", how="left")
df_gastro_total = gastro.drop(columns=COLS_ELIMINAR)




# ── One-hot Entorno (multi-label: "Costa Vasca,San Sebastián") ────────────────
entorno_series = (df_gastro_total["Entorno"].fillna("").astype(str)
                  .str.split(",").apply(lambda v: [x.strip() for x in v if x.strip()]))
entorno_dummies = entorno_series.str.join("|").str.get_dummies(sep="|").add_prefix("Entorno_")

df_gastro_total = pd.get_dummies(df_gastro_total,
    columns=["Provincia","Tipo de lugar","Nivel precio"],
    prefix=["Provincia","Tipo de lugar","Nivel precio"])
df_gastro_total = pd.concat(
    [df_gastro_total.drop(columns=["Entorno"]), entorno_dummies], axis=1
)

print(f"df_gastro_total: {df_gastro_total.shape}")
print(f"Columnas: {list(df_gastro_total.columns)}")

KeyError: 'gastro_id'

In [ ]:
df_gastro_total

,gastro_id,Calidad,valoracion,num_resenas,Michelin,Repsol,Denominación de Origen,Agricultura Ecológica,Euskal Baserri,Eusko Label,...,Tipo de lugar_tienda,Nivel precio_Caro,Nivel precio_Moderado,Nivel precio_Muy caro,Entorno_Bilbao,Entorno_Costa Vasca,Entorno_Montes y Valles vascos,Entorno_Rioja Alavesa,Entorno_San Sebastián,Entorno_Vitoria-Gasteiz
0,0,0,4.6,571.0,0,0,0,0,0,0,...,False,False,True,False,0,1,0,0,1,0
1,1,0,4.7,435.0,0,0,0,0,0,0,...,False,False,True,False,1,0,0,0,0,0
2,2,1,4.5,1925.0,1,1,0,0,0,0,...,False,False,False,True,0,0,0,0,1,0
3,3,1,4.6,1098.0,1,1,0,0,0,0,...,False,True,False,False,0,1,0,0,0,0
4,4,1,4.6,1924.0,0,0,0,0,0,0,...,False,True,False,False,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,765,0,5.0,45.0,0,0,0,0,0,1,...,True,False,True,False,0,0,1,0,0,0
486,767,1,4.6,70.0,0,0,1,0,0,0,...,True,False,True,False,0,0,1,0,0,0
487,772,1,4.3,186.0,0,0,1,0,0,0,...,True,False,True,False,0,0,0,0,0,1
488,773,1,4.2,106.0,0,0,1,0,0,0,...,True,False,True,False,0,0,0,0,0,1


In [ ]:
# ── Usuarios: todos los intereses (tu proceso exacto) ────────────────────────
user_cols  = ["id_user","municipality_id","sexo","age"]
base_users = usuarios[user_cols].drop_duplicates().copy()

intereses_por_usuario = (
    int_usu.merge(intereses[["id_interes","nombre"]], on="id_interes", how="left")
    .drop(columns=["id_interes"])
)
df_user_total = (
    base_users.merge(intereses_por_usuario, on="id_user", how="left")
    .pivot_table(index=user_cols, columns="nombre", aggfunc="size", fill_value=0)
    .reset_index()
)
df_user_total.columns.name = None
df_user_total = (df_user_total.set_index(user_cols)
                 .reindex(base_users.set_index(user_cols).index, fill_value=0).reset_index())
interest_cols = [c for c in df_user_total.columns if c not in user_cols]
df_user_total[interest_cols] = df_user_total[interest_cols].astype(int)
df_user_total["municipality_id"] = df_user_total["municipality_id"].astype(str).str[:2]
df_user_total = pd.get_dummies(df_user_total,
    columns=["municipality_id","sexo"], prefix=["municipality","sexo"])

print(f"df_user_total: {df_user_total.shape}")

df_user_total: (2000, 34)


## 4. Generación de reseñas sintéticas
Si ya tienes el archivo `data/gastro_reviews.csv`, salta esta sección.
En caso contrario, genera las reseñas con la misma lógica que patrimonio:
- σ=0.5 (señal fuerte) + 15% usuarios críticos (distribución mixta)
- Afinidad: interés gastronómico del usuario × tipo de lugar
- Usuarios con intereses gastronómicos tienen mayor probabilidad de reseñar

In [ ]:
# Mapa de afinidad: interés (id) → tipos de lugar gastronómico
AFINIDAD_GASTRO = {
    14: ["Restaurante","Asador","Sidreria"],   # Hostelería
    15: ["queseria"],                           # Queserías
    16: ["Bodega","Bodega Txakoli","tienda"],   # Gourmet
    17: ["Restaurante"],                        # Restaurante
    18: ["Asador"],                             # Asador
    19: ["Sidreria"],                           # Sidrería
    20: ["Bodega","Bodega Txakoli"],            # Bodegas
    21: ["tienda"],                             # Agricultura ecológica
    22: ["Bodega","Bodega Txakoli","tienda"],   # Denominación de Origen
    23: ["tienda"],                             # Eusko Label
    24: ["tienda"],                             # Euskal Baserri
}
INTERESES_GASTRO_SET = set(AFINIDAD_GASTRO.keys())
user_interests = int_usu.groupby("id_user")["id_interes"].apply(set).to_dict()

def tiene_interes_gastro(uid):
    return bool(user_interests.get(uid, set()) & INTERESES_GASTRO_SET)

def afinidad_gastro(uid, tipo_lugar):
    iu = user_interests.get(uid, set())
    score = sum(1 for iid, tipos in AFINIDAD_GASTRO.items()
                if iid in iu and tipo_lugar in tipos)
    return min(score, 3) / 3.0

def generar_puntuacion(uid, tipo_lugar, valoracion_real):
    base  = valoracion_real if not np.isnan(valoracion_real) else 4.0
    media = np.clip(base + afinidad_gastro(uid, tipo_lugar)*0.8 - 0.3, 1.5, 5.0)
    if random.random() < 0.85:
        p = np.random.normal(media, 0.5)
    else:
        p = np.random.normal(media - 1.5, 0.7)
    return int(np.clip(round(p), 1, 5))

TEXTOS = {
    5: ["Experiencia gastronómica excepcional, volveré sin duda.",
        "La mejor comida del País Vasco, sin exagerar.",
        "Producto de primera calidad, servicio impecable.",
        "Un lujo para el paladar, altamente recomendable.",
        "De los mejores que he probado, una joya."],
    4: ["Muy buena relación calidad-precio.",
        "Rico y bien elaborado, lo recomiendo.",
        "Buena experiencia, repetiría.",
        "Producto fresco y bien tratado, muy satisfecho.",
        "Nos fue muy bien, cocina honesta y sabrosa."],
    3: ["Correcto, sin más. Esperaba algo más.",
        "Bien pero sin sorpresas, precio algo elevado.",
        "Aceptable, hay opciones mejores en la zona.",
        "Normal, la calidad era buena pero el servicio flojo.",
        "Ni mal ni especialmente bien."],
    2: ["Decepcionante para el precio que cobran.",
        "No cumplió expectativas, he comido mejor.",
        "El producto podría ser más fresco.",
        "Servicio lento y actitud mejorable.",
        "No volvería, hay mejores opciones cerca."],
    1: ["Pésima experiencia, no lo recomiendo.",
        "El peor restaurante de la zona.",
        "Producto de baja calidad y precio abusivo.",
        "Una decepción total, nos dejaron mal sabor de boca.",
        "Evitadlo, no merece la visita."],
}

# Pesos por lugar (popularidad)
gastro["peso"] = (gastro["valoracion"].fillna(3.5)*0.6 +
                  np.log1p(gastro["num_resenas"].fillna(10))*0.4)
gastro["peso"] = gastro["peso"] / gastro["peso"].sum()

peso_usuarios = {uid: (2.5 if tiene_interes_gastro(uid) else 1.0)
                 for uid in usuarios["id_user"]}
usuarios_list = usuarios["id_user"].tolist()

reviews, pares_vistos = [], set()
fecha_inicio = datetime(2024, 1, 1)
fecha_fin    = datetime(2026, 5, 31)

for _, lugar in gastro.iterrows():
    gid   = int(lugar["gastro_id"])
    tipo  = lugar["Tipo de lugar"]
    val   = lugar["valoracion"]
    n     = int(np.clip(np.random.normal(10 + lugar["peso"]*1500, 5), 10, 50))
    cands = [u for u in usuarios_list if (u, gid) not in pares_vistos]
    if len(cands) < n: n = len(cands)
    pesos_c = np.array([peso_usuarios[u] for u in cands], dtype=float)
    pesos_c /= pesos_c.sum()
    for uid in np.random.choice(cands, size=n, replace=False, p=pesos_c):
        pares_vistos.add((uid, gid))
        p = generar_puntuacion(uid, tipo, val)
        reviews.append({
            "user_id":    int(uid),
            "gastro_id":  gid,
            "puntuacion": p,
            "texto":      None if random.random()<0.35 else random.choice(TEXTOS[p]),
            "created_at": (fecha_inicio + timedelta(
                days=random.randint(0,(fecha_fin-fecha_inicio).days))
            ).strftime("%Y-%m-%d %H:%M:%S"),
        })

df_reviews = pd.DataFrame(reviews)
print(f"Reseñas generadas: {len(df_reviews)}")
print(f"Usuarios únicos:   {df_reviews['user_id'].nunique()}")
print(f"Lugares únicos:    {df_reviews['gastro_id'].nunique()}")
print(f"\nDistribución:")
dist = df_reviews["puntuacion"].value_counts(normalize=True).sort_index()*100
for p, pct in dist.items():
    print(f"  {p}★  {'█'*int(pct/2)}  {pct:.1f}%")
print(f"\nMedia: {df_reviews['puntuacion'].mean():.3f}  Std: {df_reviews['puntuacion'].std():.3f}")

os.makedirs("data", exist_ok=True)
df_reviews.to_csv("data/gastro_reviews.csv", index=False)
print("\n✅ data/gastro_reviews.csv guardado")

Reseñas generadas: 6581
Usuarios únicos:   1859
Lugares únicos:    490

Distribución:
  1★    0.2%
  2★  █  3.6%
  3★  █████  11.4%
  4★  ████████████████████  40.9%
  5★  █████████████████████  43.7%

Media: 4.243  Std: 0.813

✅ data/gastro_reviews.csv guardado


## 5. SVD — Filtrado colaborativo

In [ ]:
# Si ya tienes el CSV, cárgalo directamente:
df_reviews = pd.read_csv("../data/reviews_gastronomy.csv")

reader = Reader(rating_scale=(1, 5))
data   = Dataset.load_from_df(df_reviews[["user_id","gastro_id","puntuacion"]], reader)

# Comparativa de algoritmos
algoritmos = {
    "BaselineOnly":  BaselineOnly(),
    "KNNWithMeans":  KNNWithMeans(k=40, sim_options={"name":"pearson","user_based":True}),
    "SVD":           SVD(n_factors=50, n_epochs=30, random_state=42),
    "NMF":           NMF(n_factors=15, random_state=42),
}

print(f"{'Algoritmo':<15}  {'RMSE':>6}  {'±':>6}  {'MAE':>6}")
print("─"*42)
for nombre, algo in algoritmos.items():
    cv = cross_validate(algo, data, measures=["RMSE","MAE"], cv=5, verbose=False)
    print(f"{nombre:<15}  {cv['test_rmse'].mean():>6.4f}  "
          f"{cv['test_rmse'].std():>6.4f}  {cv['test_mae'].mean():>6.4f}")

Algoritmo          RMSE       ±     MAE
──────────────────────────────────────────
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
BaselineOnly     1.0210  0.0113  0.7768
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
KNNWithMeans     1.1343  0.0227  0.8674
SVD              1.0461  0.0137  0.8032
NMF              1.2277  0.0096  0.9706


In [ ]:
# Entrenar SVD con todo el dataset
full_trainset = data.build_full_trainset()
model_svd = SVD(n_factors=50, n_epochs=30, random_state=42)
model_svd.fit(full_trainset)

print(f"SVD entrenado con {full_trainset.n_users} usuarios y {full_trainset.n_items} lugares.")

SVD entrenado con 1931 usuarios y 490 lugares.


## 6. Modelo de contenido — Feature engineering gastronómico

Usa tu feature engineering completo con **afinidad extendida** para gastronomía.
Sirve principalmente para el **cold start** (usuarios sin historial).

In [ ]:
# ── Join reviews ← usuarios ← gastro_total ──────────────────────────────────
df_fe = (
    df_reviews.drop(columns=["texto","created_at"])
    .merge(df_user_total, left_on="user_id",   right_on="id_user",   how="left")
    .merge(df_gastro_total, on="gastro_id", how="left")
)

# ── Afinidad extendida usuario × tipo de lugar gastronómico ──────────────────
AFINIDAD_FE = {
    "Hostelería":             "Tipo de lugar_Restaurante",
    "Restaurante":            "Tipo de lugar_Restaurante",
    "Asador":                 "Tipo de lugar_Asador",
    "Sidrería":               "Tipo de lugar_Sidreria",
    "Bodegas":                "Tipo de lugar_Bodega",
    "Queserías":              "Tipo de lugar_queseria",
    "Gourmet":                "Tipo de lugar_tienda",
    "Agricultura ecológica":  "Tipo de lugar_tienda",
    "Denominación de Origen": "Tipo de lugar_Bodega",
    "Eusko Label":            "Tipo de lugar_tienda",
    "Euskal Baserri":         "Tipo de lugar_tienda",
}
df_fe["afinidad"] = sum(
    df_fe[u].fillna(0) * df_fe[l].fillna(0)
    for u, l in AFINIDAD_FE.items()
    if u in df_fe.columns and l in df_fe.columns
).astype(float)

# ── MinMaxScaler valoracion (4.1–5.0 → 1.0–5.0) ─────────────────────────────
df_fe["valoracion"] = (df_fe.groupby("gastro_id")["valoracion"]
                        .transform(lambda x: x.fillna(x.median()))
                        .fillna(df_fe["valoracion"].median()))
scaler_val = MinMaxScaler(feature_range=(1, 5))
df_fe["valoracion_scaled"] = scaler_val.fit_transform(df_fe[["valoracion"]])
df_fe.drop(columns=["valoracion"], inplace=True)

# ── X / y ─────────────────────────────────────────────────────────────────────
DROP = ["user_id","id_user","gastro_id","puntuacion"]
X_fe = df_fe.drop(columns=[c for c in DROP if c in df_fe.columns])
X_fe = X_fe.select_dtypes(include=["number","bool"]).copy()
X_fe[X_fe.select_dtypes("bool").columns] = X_fe.select_dtypes("bool").astype(int)
y    = df_fe["puntuacion"]

print(f"X_fe shape: {X_fe.shape}  |  NaNs: {X_fe.isna().any().sum()}")
print(f"Features clave: {[c for c in X_fe.columns if c in ['afinidad','valoracion_scaled']]}")

X_fe shape: (15581, 63)  |  NaNs: 0
Features clave: ['afinidad', 'valoracion_scaled']


In [ ]:
# Entrenar modelo de contenido y comparar con SVD
idx_train, idx_test = sk_split(np.arange(len(df_reviews)), test_size=0.2, random_state=42)

pipe_content = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   GradientBoostingRegressor(random_state=42))
])
pipe_content.fit(X_fe.iloc[idx_train], y.iloc[idx_train])

df_test        = df_reviews.iloc[idx_test]
svd_scores     = df_test.apply(
    lambda r: model_svd.predict(r["user_id"], r["gastro_id"]).est, axis=1).values
content_scores = pipe_content.predict(X_fe.iloc[idx_test])
y_test_vals    = y.iloc[idx_test].values

rmse_svd     = np.sqrt(mean_squared_error(y_test_vals, svd_scores))
rmse_content = np.sqrt(mean_squared_error(y_test_vals, content_scores))
print(f"SVD puro        RMSE={rmse_svd:.4f}  MAE={mean_absolute_error(y_test_vals, svd_scores):.4f}")
print(f"Contenido puro  RMSE={rmse_content:.4f}  MAE={mean_absolute_error(y_test_vals, content_scores):.4f}")
print()
print("ℹ️  SVD es el modelo principal. Contenido se usa para cold start.")

SVD puro        RMSE=0.7595  MAE=0.5861
Contenido puro  RMSE=0.9900  MAE=0.7456

ℹ️  SVD es el modelo principal. Contenido se usa para cold start.


## 7. Función de recomendación gastronómica

In [ ]:
def recomendar_gastro(user_id, model_svd, df_reviews, gastro,
                      tipo_lugar=None, provincia=None, precio=None, top_n=10):
    """
    Ranking personalizado de servicios gastronómicos para un usuario.

    Parámetros:
        user_id:     ID del usuario
        tipo_lugar:  filtro opcional ('Restaurante','Bodega','Sidreria','Asador',
                                      'Bodega Txakoli','queseria','tienda')
        provincia:   filtro opcional ('Guipúzcoa','Vizcaya','Álava')
        precio:      filtro opcional ('Moderado','Caro','Muy caro')
        top_n:       número de recomendaciones

    Retorna:
        DataFrame ordenado con puntuacion_estimada
    """
    tiene_historial = user_id in df_reviews["user_id"].values
    visitados = set(df_reviews[df_reviews["user_id"]==user_id]["gastro_id"])

    # Aplicar filtros sobre el catálogo
    catalogo = df_gastro_total.copy()
    if tipo_lugar: catalogo = catalogo[catalogo["Tipo de lugar"] == tipo_lugar]
    if provincia:  catalogo = catalogo[catalogo["Provincia"] == provincia]
    if precio:     catalogo = catalogo[catalogo["Nivel precio"] == precio]

    no_visitados = [gid for gid in catalogo["gastro_id"] if gid not in visitados]

    if tiene_historial:
        preds = [(gid, model_svd.predict(user_id, gid).est) for gid in no_visitados]
        metodo = "SVD personalizado"
    else:
        # Cold start: ordenar por valoracion real (Google Maps)
        media_global = df_reviews.groupby("gastro_id")["puntuacion"].mean()
        preds = [(gid, media_global.get(gid, catalogo[catalogo["gastro_id"]==gid]["valoracion"].values[0]))
                 for gid in no_visitados]
        metodo = "Cold start (valoración media)"

    preds.sort(key=lambda x: x[1], reverse=True)

    df_top = pd.DataFrame(preds[:top_n], columns=["gastro_id","puntuacion_estimada"])
    df_top = df_top.merge(
        df_gastro_total[["gastro_id","Nombre","Tipo de lugar","Provincia","Nivel precio",
                         "valoracion","num_resenas","Michelin","Repsol"]],
        on="gastro_id", how="left"
    )
    df_top["puntuacion_estimada"] = df_top["puntuacion_estimada"].round(2)
    df_top["metodo"] = metodo
    return df_top.reset_index(drop=True)

print("Función recomendar_gastro definida.")

Función recomendar_gastro definida.


## 8. Ejemplos de ranking

In [ ]:
# Top 10 general para un usuario
uid = int(df_reviews["user_id"].iloc[0])
ranking = recomendar_gastro(uid, model_svd, df_reviews, gastro, top_n=10)

print(f"Top 10 para usuario {uid} ({ranking['metodo'].iloc[0]}):\n")
for i, row in ranking.iterrows():
    michelin = "⭐" if row["Michelin"] else ""
    repsol   = "🔵" if row["Repsol"]   else ""
    print(f"  {i+1:>2}.  {row['puntuacion_estimada']:.2f}  "
          f"{str(row['Nombre'])[:35]:<35}  "
          f"[{row['Tipo de lugar']}]  {row['Nivel precio']}  {michelin}{repsol}")
print()
ranking

KeyError: "['Nombre', 'Tipo de lugar', 'Provincia', 'Nivel precio'] not in index"

In [ ]:
# Top 5 Bodegas en Álava (Rioja Alavesa)
print("=== Top 5 Bodegas en Álava ===\n")
ranking_bodegas = recomendar_gastro(
    uid, model_svd, df_reviews, gastro,
    tipo_lugar="Bodega", provincia="Álava", top_n=5
)
for i, row in ranking_bodegas.iterrows():
    print(f"  {i+1}. {row['puntuacion_estimada']:.2f}  {row['Nombre']}")

# Top 5 Sidrerías
print("\n=== Top 5 Sidrerías ===\n")
ranking_sidrerias = recomendar_gastro(
    uid, model_svd, df_reviews, gastro,
    tipo_lugar="Sidreria", top_n=5
)
for i, row in ranking_sidrerias.iterrows():
    print(f"  {i+1}. {row['puntuacion_estimada']:.2f}  {row['Nombre']}")

In [ ]:
# Usuario nuevo (cold start)
ranking_nuevo = recomendar_gastro(99999, model_svd, df_reviews, gastro, top_n=5)
print(f"Top 5 para usuario nuevo ({ranking_nuevo['metodo'].iloc[0]}):\n")
for i, row in ranking_nuevo.iterrows():
    print(f"  {i+1}. {row['puntuacion_estimada']:.2f}  {row['Nombre']}  [{row['Tipo de lugar']}]")

## 9. Guardar modelos

In [ ]:
os.makedirs("ML/models", exist_ok=True)

with open("ML/models/svd_gastro.pkl",       "wb") as f: pickle.dump(model_svd,    f)
with open("ML/models/content_gastro.pkl",   "wb") as f: pickle.dump(pipe_content, f)
with open("ML/models/scaler_gastro.pkl",    "wb") as f: pickle.dump(scaler_val,   f)
pd.Series(X_fe.columns.tolist()).to_csv("ML/models/feature_cols_gastro.csv", index=False)

print("Modelos guardados en ML/models/:")
print("  svd_gastro.pkl        ← modelo principal SVD")
print("  content_gastro.pkl    ← modelo de contenido (cold start)")
print("  scaler_gastro.pkl     ← MinMaxScaler de valoracion")
print("  feature_cols_gastro   ← columnas de X_fe para inferencia")

## 10. Integración en el backend de Render

In [ ]:
# GET /get_prediction?user_id=123&categoria=gastro&tipo_lugar=Bodega&top_n=10

backend_code = '''
import pickle, pandas as pd
from flask import Flask, jsonify, request

app = Flask(__name__)

with open("ML/models/svd_gastro.pkl", "rb") as f:
    model_gastro = pickle.load(f)

df_reviews_gastro = pd.read_csv("data/gastro_reviews.csv")
gastro            = pd.read_csv("data/gastronomia.csv")
gastro.rename(columns={"Unnamed: 0": "gastro_id"}, inplace=True)

@app.route("/get_prediction")
def get_prediction():
    user_id    = int(request.args.get("user_id"))
    top_n      = int(request.args.get("top_n", 10))
    tipo_lugar = request.args.get("tipo_lugar", None)  # opcional
    provincia  = request.args.get("provincia", None)   # opcional

    tiene_historial = user_id in df_reviews_gastro["user_id"].values
    visitados = set(df_reviews_gastro[df_reviews_gastro["user_id"]==user_id]["gastro_id"])

    catalogo = gastro.copy()
    if tipo_lugar: catalogo = catalogo[catalogo["Tipo de lugar"] == tipo_lugar]
    if provincia:  catalogo = catalogo[catalogo["Provincia"] == provincia]

    no_visitados = [g for g in catalogo["gastro_id"] if g not in visitados]

    if tiene_historial:
        preds = sorted(
            [(g, model_gastro.predict(user_id, g).est) for g in no_visitados],
            key=lambda x: x[1], reverse=True
        )[:top_n]
    else:
        media = df_reviews_gastro.groupby("gastro_id")["puntuacion"].mean()
        preds = sorted(
            [(g, media.get(g, gastro[gastro["gastro_id"]==g]["valoracion"].values[0]))
             for g in no_visitados],
            key=lambda x: x[1], reverse=True
        )[:top_n]

    resultado = []
    for gid, score in preds:
        row = gastro[gastro["gastro_id"]==gid].iloc[0]
        resultado.append({
            "gastro_id":           int(gid),
            "nombre":              row["Nombre"],
            "tipo_lugar":          row["Tipo de lugar"],
            "provincia":           row["Provincia"],
            "nivel_precio":        row["Nivel precio"],
            "michelin":            bool(row.get("Michelin", 0)),
            "repsol":              bool(row.get("Repsol", 0)),
            "puntuacion_estimada": round(float(score), 2)
        })

    return jsonify({"user_id": user_id, "recomendaciones": resultado})
'''
print(backend_code)